# TumorClassifier Pipeline

This notebook uses the reorganized project modules. Set `TUMOR_DATA_ROOT` if your data is not in `./data`.

In [ ]:
import os
from pathlib import Path

from tumor_classifier.config import DataPaths, TrainingConfig
from tumor_classifier.data.matlab_conversion import convert_matlab_directory
from tumor_classifier.data.image_processing import process_directory
from tumor_classifier.data.dataset_utils import merge_datasets
from tumor_classifier.data.augmentation import augment_nontumor_images, augment_tumor_images
from tumor_classifier.models.classifier import Classifier
from tumor_classifier.models.svm_baseline import TumourClassifier
from tumor_classifier.training.dataloaders import get_data_loaders, load_saved_features, save_features
from tumor_classifier.training.feature_extraction import compute_alexnet_features, load_alexnet
from tumor_classifier.training.train import train_net
from tumor_classifier.training.evaluate import evaluate, get_model_name
from tumor_classifier.utils.device import get_device

paths = DataPaths()
device = get_device()
print(f"Data root: {paths.root}")
print(f"Device: {device}")

## 1. Data processing

In [ ]:
# convert_matlab_directory(paths.matlab_raw, paths.matlab_processed)
# process_directory(paths.kaggle_healthy_raw, paths.kaggle_healthy_224)
# merge_datasets(paths)

## 2. Augmentation

In [ ]:
import torchvision.transforms as transforms
import torchvision

train_data = torchvision.datasets.ImageFolder(
    root=str(paths.dataset_split / "train"),
    transform=transforms.ToTensor(),
)

# augment_nontumor_images(train_data, paths.augmented / "NoTumor")
# augment_tumor_images(train_data, paths.augmented / "Tumor")

## 3. Primary model (AlexNet + classifier head)

In [ ]:
train_loader, val_loader, test_loader, classes = get_data_loaders(256, paths)
alexnet = load_alexnet(device)

train_features, train_labels = compute_alexnet_features(train_loader, alexnet, device)
val_features, val_labels = compute_alexnet_features(val_loader, alexnet, device)
test_features, test_labels = compute_alexnet_features(test_loader, alexnet, device)

save_features(
    paths.feature_maps,
    train_features,
    val_features,
    test_features,
    train_labels,
    val_labels,
    test_labels,
)

In [ ]:
config = TrainingConfig(batch_size=32, num_epochs=30)
feature_train_loader, feature_val_loader, feature_test_loader = load_saved_features(
    paths.feature_maps,
    batch_size=config.batch_size,
)

classifier = Classifier().to(device)
train_net(classifier, feature_train_loader, feature_val_loader, device, config)

## 4. Baseline SVM model

In [ ]:
from torch.utils.data import DataLoader

train_data = torchvision.datasets.ImageFolder(
    root=str(paths.augmented),
    transform=transforms.ToTensor(),
)
test_data = torchvision.datasets.ImageFolder(
    root=str(paths.dataset_split / "test"),
    transform=transforms.ToTensor(),
)

svm_train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
svm_test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

svm_classifier = TumourClassifier()
svm_classifier.train_svm(svm_train_loader)
svm_classifier.evaluate(svm_test_loader)